# COMP4381 Phase 4: Shared Modeling Dataset

**Author responsibility:** Ahmad — modeling dataset and feature-engineering foundation only.

## TL;DR

This notebook prepares the common airport-day dataset for a later two-stage prediction system. The Phase 3 input contains **16,069 rows, 35 columns, 5 countries, and 10 airports**, with no duplicate rows. After constructing leakage-safe lag features and removing their initial unavailable rows, the shared modeling dataset contains **15,999 rows and 15 features**. No classification or regression model is trained here.

## 1. Context and Methods

Phase 4 uses a two-stage design:

1. **Classification (Kamel):** predict whether an airport-day is delayed using `target_delayed_15`.
2. **Regression (Yusra):** for delayed airport-days, predict expected arrival-delay minutes using `target_delay_minutes`.

### Key assumptions

- Each row represents one airport-day.
- Historical delay outcomes are available before predicting a later airport-day.
- Rows must be split chronologically so later observations do not influence training.
- This notebook creates shared data and preprocessing only; it deliberately does not fit a predictive model.

## 2. Setup

Import the course-level data and preprocessing libraries, then define paths that work when the notebook is launched from either the project root or the `notebooks` directory.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "phase3_airport_day_clean.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "phase3_airport_day_clean.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "phase4_modeling_dataset.csv"

assert INPUT_PATH.exists(), f"Input file not found: {INPUT_PATH}"
print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")

Input:  /Users/ahmad/Downloads/COMP4381_Flight_Delay_Project-main/data/processed/phase3_airport_day_clean.csv
Output: /Users/ahmad/Downloads/COMP4381_Flight_Delay_Project-main/data/processed/phase4_modeling_dataset.csv


## 3. Load the Phase 3 Dataset

Load the cleaned airport-day CSV with Pandas. The shape reports the number of airport-day records and available columns; the full column list makes the modeling inputs auditable.

In [2]:
df = pd.read_csv(INPUT_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nColumn names:")
for column_number, column_name in enumerate(df.columns, start=1):
    print(f"{column_number:>2}. {column_name}")

Dataset shape: 16,069 rows x 35 columns

Column names:
 1. Date
 2. Airport
 3. Departure Punctuality %
 4. Arrival Punctuality %
 5. Avg Departure Schedule Delay
 6. Avg Arrival Schedule Delay
 7. Avg Departure - Arrival Schedule Delay
 8. Operated Schedules %
 9. airport_norm
10. id
11. name
12. municipality
13. iso_country
14. iata_code
15. ident
16. latitude_deg
17. longitude_deg
18. elevation_ft
19. type
20. scheduled_service
21. score
22. date
23. year
24. month
25. weekday
26. is_weekend
27. season
28. dep_punctuality_pct
29. arr_punctuality_pct
30. dep_delay_min
31. arr_delay_min
32. dep_minus_arr_delay_min
33. target_delay_minutes
34. target_delayed_15
35. target_delay_category


## 4. Convert and Inspect the Date Field

The project uses the engineered lowercase `date` column as its modeling timestamp. Converting it to `datetime64` enables reliable sorting, date-range checks, and chronological splitting. The inspection below also checks data types, missingness, duplicates, geographic coverage, airport coverage, and time coverage.

In [3]:
assert "date" in df.columns, "Required modeling date column 'date' is missing."
df["date"] = pd.to_datetime(df["date"], errors="raise")

print("Dataset information:")
df.info()

missing_values = df.isna().sum()
duplicate_count = int(df.duplicated().sum())
country_count = int(df["iso_country"].nunique(dropna=True))
airport_count = int(df["Airport"].nunique(dropna=True))
date_min = df["date"].min()
date_max = df["date"].max()

print("\nMissing values per column:")
print(missing_values.to_string())
print(f"\nDuplicate rows: {duplicate_count:,}")
print(f"Number of countries: {country_count:,}")
print(f"Number of airports: {airport_count:,}")
print(f"Date range: {date_min.date()} to {date_max.date()}")

if duplicate_count == 0:
    print("Duplicate check passed: every airport-day row is unique as a complete record.")
else:
    print("Duplicate rows were found and should be investigated before modeling.")

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 16069 entries, 0 to 16068
Data columns (total 35 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   Date                                    16069 non-null  str           
 1   Airport                                 16069 non-null  str           
 2   Departure Punctuality %                 16069 non-null  float64       
 3   Arrival Punctuality %                   16069 non-null  float64       
 4   Avg Departure Schedule Delay            16069 non-null  float64       
 5   Avg Arrival Schedule Delay              16069 non-null  float64       
 6   Avg Departure - Arrival Schedule Delay  16069 non-null  float64       
 7   Operated Schedules %                    16069 non-null  float64       
 8   airport_norm                            16069 non-null  str           
 9   id                                      

## 5. Validate Project Requirements

The assertions turn the Phase 4 assignment rules into reproducible checks. Here, the 35 source columns include identifiers, geography, airport properties, calendar fields, operational measurements, and targets, so they exceed the requirement for at least 15 meaningful columns.

In [4]:
assert len(df) >= 1000, "Requirement failed: dataset must have at least 1,000 rows."
assert country_count >= 5, "Requirement failed: dataset must cover at least 5 countries."
assert df.shape[1] >= 15, "Requirement failed: dataset must have at least 15 meaningful columns."
assert duplicate_count == 0, "Requirement failed: duplicate rows must be resolved or explained."

print("All Phase 4 dataset requirements passed:")
print(f"- Rows: {len(df):,} (minimum 1,000)")
print(f"- Countries: {country_count} (minimum 5)")
print(f"- Meaningful source columns: {df.shape[1]} (minimum 15)")
print(f"- Duplicate rows: {duplicate_count}")

All Phase 4 dataset requirements passed:
- Rows: 16,069 (minimum 1,000)
- Countries: 5 (minimum 5)
- Meaningful source columns: 35 (minimum 15)
- Duplicate rows: 0


## 6. Sort Airport Histories and Define Targets

Lag features must follow the correct sequence within each airport, so rows are sorted by `Airport` and `date`. The two targets are validated but remain separate from the predictor list.

In [5]:
df = df.sort_values(["Airport", "date"]).reset_index(drop=True)

target_class = "target_delayed_15"
target_reg = "target_delay_minutes"

assert target_class in df.columns, f"Missing classification target: {target_class}"
assert target_reg in df.columns, f"Missing regression target: {target_reg}"
assert set(df[target_class].dropna().unique()).issubset({0, 1}), \
    "The classification target must contain only 0 and 1."
assert df[[target_class, target_reg]].notna().all().all(), "Target columns contain missing values."

print(f"Classification target: {target_class}")
print(f"Regression target: {target_reg}")

Classification target: target_delayed_15
Regression target: target_delay_minutes


## 7. Identify Leakage Columns

Target values and same-day arrival-delay or punctuality measurements would reveal the answer during prediction. They are therefore treated as leakage and excluded from `feature_cols`. The list is filtered against the actual dataset columns so an optional name can be absent without causing an error.

In [6]:
leakage_candidates = [
    "target_delay_minutes",
    "target_delayed_15",
    "target_delay_category",
    "arr_delay_min",
    "Avg Arrival Schedule Delay",
    "Arrival Punctuality %",
    "arr_punctuality_pct",
]

leakage_cols = [column for column in leakage_candidates if column in df.columns]
print("Leakage columns found and excluded:")
print(leakage_cols)

Leakage columns found and excluded:
['target_delay_minutes', 'target_delayed_15', 'target_delay_category', 'arr_delay_min', 'Avg Arrival Schedule Delay', 'Arrival Punctuality %', 'arr_punctuality_pct']


## 8. Create Historical Delay Features

For each airport, the model may use known outcomes from earlier airport-days. `shift(1)` guarantees that the rolling average starts with the previous record rather than the current target. The 7-record lag and rolling window are record-based within each airport, as required.

In [7]:
airport_delay_groups = df.groupby("Airport")[target_reg]

df["prev_arr_delay_1d"] = airport_delay_groups.shift(1)
df["prev_arr_delay_7d"] = airport_delay_groups.shift(7)
df["rolling_arr_delay_7d"] = airport_delay_groups.transform(
    lambda airport_values: airport_values.shift(1).rolling(window=7, min_periods=7).mean()
)

lag_cols = ["prev_arr_delay_1d", "prev_arr_delay_7d", "rolling_arr_delay_7d"]
print("Missing lag values before dropping rows:")
print(df[lag_cols].isna().sum().to_string())

rows_before_lag_drop = len(df)
df_model = df.dropna(subset=lag_cols).copy()
rows_dropped_for_lags = rows_before_lag_drop - len(df_model)

assert df_model[lag_cols].notna().all().all(), "Lag features still contain missing values."
print(f"\nRows dropped because prior history was unavailable: {rows_dropped_for_lags:,}")
print(f"Rows remaining for modeling: {len(df_model):,}")

Missing lag values before dropping rows:
prev_arr_delay_1d       10
prev_arr_delay_7d       70
rolling_arr_delay_7d    70

Rows dropped because prior history was unavailable: 70
Rows remaining for modeling: 15,999


## 9. Select the Shared Feature Set

The shared predictors combine calendar context, country and airport identity, airport characteristics, and historical delays. Only columns present in the dataset are retained, and a final assertion guarantees that leakage columns are absent.

In [8]:
suggested_features = [
    "year",
    "month",
    "weekday",
    "is_weekend",
    "season",
    "iso_country",
    "Airport",
    "type",
    "scheduled_service",
    "latitude_deg",
    "longitude_deg",
    "elevation_ft",
    "prev_arr_delay_1d",
    "prev_arr_delay_7d",
    "rolling_arr_delay_7d",
]

feature_cols = [
    column for column in suggested_features
    if column in df_model.columns and column not in leakage_cols
]

assert feature_cols, "No usable modeling features were found."
assert not set(feature_cols).intersection(leakage_cols), "Leakage detected in feature_cols."
assert target_class not in feature_cols and target_reg not in feature_cols

print(f"Selected {len(feature_cols)} shared features:")
print(feature_cols)

Selected 15 shared features:
['year', 'month', 'weekday', 'is_weekend', 'season', 'iso_country', 'Airport', 'type', 'scheduled_service', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'prev_arr_delay_1d', 'prev_arr_delay_7d', 'rolling_arr_delay_7d']


## 10. Create Modeling Variables and a Chronological 80/20 Split

The modeling rows are sorted globally by date. The first 80% form the training period and the last 20% form the test period. This preserves time order and avoids the optimistic evaluation that can result from randomly mixing past and future airport-days.

In [9]:
df_model = df_model.sort_values(["date", "Airport"]).reset_index(drop=True)

X = df_model[feature_cols]
y_class = df_model["target_delayed_15"]
y_reg = df_model["target_delay_minutes"]

split_index = int(len(df_model) * 0.80)
assert 0 < split_index < len(df_model), "The dataset is too small for an 80/20 split."

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_class_train = y_class.iloc[:split_index].copy()
y_class_test = y_class.iloc[split_index:].copy()
y_reg_train = y_reg.iloc[:split_index].copy()
y_reg_test = y_reg.iloc[split_index:].copy()

train_dates = df_model["date"].iloc[:split_index]
test_dates = df_model["date"].iloc[split_index:]
assert train_dates.max() <= test_dates.min(), "Chronological split order is incorrect."

print(f"Training period: {train_dates.min().date()} to {train_dates.max().date()}")
print(f"Test period:     {test_dates.min().date()} to {test_dates.max().date()}")
print("\nTrain/test output shapes:")
print(f"X_train:       {X_train.shape}")
print(f"X_test:        {X_test.shape}")
print(f"y_class_train: {y_class_train.shape}")
print(f"y_class_test:  {y_class_test.shape}")
print(f"y_reg_train:   {y_reg_train.shape}")
print(f"y_reg_test:    {y_reg_test.shape}")

Training period: 2022-01-08 to 2025-07-10
Test period:     2025-07-11 to 2026-05-26

Train/test output shapes:
X_train:       (12799, 15)
X_test:        (3200, 15)
y_class_train: (12799,)
y_class_test:  (3200,)
y_reg_train:   (12799,)
y_reg_test:    (3200,)


## 11. Build and Test the Shared Preprocessor

Numeric features use median imputation followed by standardization. Categorical features use most-frequent imputation followed by one-hot encoding, with unseen test categories ignored safely. Fitting occurs on `X_train` only; `X_test` is transformed with the learned training rules. This tests the preprocessing foundation without training a model.

In [10]:
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = [column for column in feature_cols if column not in numeric_cols]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

assert X_train_prepared.shape[0] == X_train.shape[0]
assert X_test_prepared.shape[0] == X_test.shape[0]
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Prepared training matrix shape: {X_train_prepared.shape}")
print(f"Prepared test matrix shape:     {X_test_prepared.shape}")
print("Preprocessor test passed; no predictive model was fitted.")

Numeric columns (10): ['year', 'month', 'weekday', 'is_weekend', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'prev_arr_delay_1d', 'prev_arr_delay_7d', 'rolling_arr_delay_7d']
Categorical columns (5): ['season', 'iso_country', 'Airport', 'type', 'scheduled_service']
Prepared training matrix shape: (12799, 33)
Prepared test matrix shape:     (3200, 33)
Preprocessor test passed; no predictive model was fitted.


## 12. Save the Shared Modeling Dataset

Save the chronological modeling base with its date, airport, selected features, and both targets. `dict.fromkeys` preserves order while avoiding a duplicate `Airport` column because `Airport` is already part of `feature_cols`.

In [11]:
save_cols = list(dict.fromkeys([
    "date",
    "Airport",
    *feature_cols,
    "target_delayed_15",
    "target_delay_minutes",
]))

df_model[save_cols].to_csv(OUTPUT_PATH, index=False)

saved_check = pd.read_csv(OUTPUT_PATH)
assert saved_check.shape == (len(df_model), len(save_cols))
assert saved_check.columns.tolist() == save_cols

print(f"Saved modeling dataset: {OUTPUT_PATH}")
print(f"Saved shape: {saved_check.shape[0]:,} rows x {saved_check.shape[1]} columns")
print("Saved columns:")
print(saved_check.columns.tolist())

Saved modeling dataset: /Users/ahmad/Downloads/COMP4381_Flight_Delay_Project-main/data/processed/phase4_modeling_dataset.csv
Saved shape: 15,999 rows x 18 columns
Saved columns:
['date', 'Airport', 'year', 'month', 'weekday', 'is_weekend', 'season', 'iso_country', 'type', 'scheduled_service', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'prev_arr_delay_1d', 'prev_arr_delay_7d', 'rolling_arr_delay_7d', 'target_delayed_15', 'target_delay_minutes']


## 13. Team Handoff Summary

- **Why leakage columns were removed:** same-day target, arrival-delay, delay-category, and arrival-punctuality fields directly reveal or closely encode the outcome. Using them as predictors would produce unrealistically strong results that would not be available at prediction time.
- **Why lag features were created:** an airport's recent delay history can carry useful operational and seasonal information. The lags and rolling average use only earlier records within that airport, never the current row's target.
- **Why chronological splitting was used:** prediction is a future-facing task. Training on the earliest 80% and testing on the latest 20% better represents real deployment and prevents future observations from leaking backward through a random split.
- **Kamel's classification input:** use `X_train` and `y_class_train` for training, then `X_test` and `y_class_test` for evaluation. Reuse the shared `preprocessor` inside Kamel's classification pipeline.
- **Yusra's regression input:** begin from the same chronological modeling base, filter to rows where `target_delayed_15 == 1`, and use `target_delay_minutes` as the regression outcome. Preserve chronological order and reuse the shared feature definitions and preprocessing pattern.

The preprocessor has been fitted and transformed only as a compatibility test. **No Logistic Regression, Linear Regression, XGBoost, or other predictive model is trained in this notebook.**